In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import os

df = pd.read_csv("data/breast-cancer.csv")
df = df.drop(columns=["id"])
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

# Create the directory if it doesn't exist
os.makedirs('models', exist_ok=True)
joblib.dump({"model": model, "scaler": scaler}, "models/breast_cancer_model.joblib")
print("Model saved.")

Accuracy: 0.9649122807017544
              precision    recall  f1-score   support

           0       0.96      0.99      0.97        71
           1       0.98      0.93      0.95        43

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114

[[70  1]
 [ 3 40]]
Model saved.


In [ ]:
import joblib
import numpy as np

bundle = joblib.load("models/breast_cancer_model.joblib")
model = bundle["model"]
scaler = bundle["scaler"]

def predict(features):
    X = np.array(features).reshape(1, -1)
    X = scaler.transform(X)
    pred = model.predict(X)[0]
    prob = model.predict_proba(X).max()
    return pred, prob

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

app = FastAPI()

bundle = joblib.load("models/breast_cancer_model.joblib")
model = bundle["model"]
scaler = bundle["scaler"]

class Input(BaseModel):
    data: list   # list of 30 features

@app.post("/predict")
def predict_cancer(input: Input):
    X = np.array(input.data).reshape(1, -1)
    X = scaler.transform(X)
    pred = int(model.predict(X)[0])
    prob = float(model.predict_proba(X).max())
    return {"prediction": pred, "confidence": prob}

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

df = pd.read_csv("data/breast-cancer.csv")
df = df.drop(columns=["id"])
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})

# 1) Bar plot
plt.figure(figsize=(6,4))
df["diagnosis"].value_counts().plot(kind="bar")
plt.xticks([0,1], ["Benign", "Malignant"])
plt.title("Diagnosis Distribution")
plt.show()

# 2) Histograms
df.hist(figsize=(18,18), bins=25)
plt.show()

# 3) Boxplot
sns.boxplot(x="diagnosis", y="radius_mean", data=df)
plt.show()

# 4) Heatmap
plt.figure(figsize=(14,12))
sns.heatmap(df.corr(), cmap="coolwarm")
plt.show()

# 5) Scatter
sns.scatterplot(x="radius_mean", y="texture_mean", hue="diagnosis", data=df)
plt.show()

# 6) 3D scatter
fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection="3d")

ax.scatter(df["radius_mean"], df["texture_mean"], df["perimeter_mean"],
           c=["red" if d==1 else "blue" for d in df["diagnosis"]])
ax.set_title("3D Scatter Plot")
plt.show()

# 7) 3D Surface
x = np.linspace(df["radius_mean"].min(), df["radius_mean"].max(), 50)
y2 = np.linspace(df["texture_mean"].min(), df["texture_mean"].max(), 50)
Xgrid, Ygrid = np.meshgrid(x, y2)
Zgrid = 0.5*Xgrid + 0.3*Ygrid

fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(Xgrid, Ygrid, Zgrid, cmap="viridis", alpha=0.4)
plt.show()

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

df = pd.read_csv("data/breast-cancer.csv")
df = df.drop(columns=["id"])
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection="3d")

ax.scatter(
    X_pca[:,0], X_pca[:,1], X_pca[:,2],
    c=["red" if c==1 else "blue" for c in y],
    alpha=0.7
)

ax.set_title("3D PCA Projection")
plt.show()


In [ ]:
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

df = pd.read_csv("data/breast-cancer.csv")
df = df.drop(columns=["id"])
df["diagnosis"] = df["diagnosis"].map({"M": "Malignant", "B": "Benign"})

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

fig = px.scatter_3d(
    x=X_pca[:,0], y=X_pca[:,1], z=X_pca[:,2],
    color=y,
    title="3D PCA Interactive Plot",
    labels={"color": "Diagnosis"}
)

fig.show()
